# 🌙 Benchmarking Coding Agent on Intel Lunar Lake

**Notebook 3 of 3** — Intel Core Ultra Series 2 (Codename: Lunar Lake)

**Target Hardware:**
- SoC: Intel Core Ultra 7 258V / Core Ultra 5 226V (Lunar Lake)
- iGPU: Arc 140V / Arc 130V (Xe2 architecture)
- NPU: NPU4 (48 TOPS, 6 compute engines)
- Memory: Unified LPDDR5X (shared CPU+GPU pool)
- OS: Ubuntu 24.04 Noble

**Key differences from Panther Lake:**
- NPU4 vs NPU5 — Lunar Lake NPU has better driver maturity
- Xe2 vs Xe3 — Panther Lake iGPU is faster
- Unified memory is important: large models consume shared RAM
- NPU officially tested on MeteorLake, **LunarLake**, ArrowLake (most stable)

---

## 1. Prerequisites & Driver Verification

In [ ]:
import sys
!{sys.executable} -m pip install -q \
    openvino openvino-genai openvino-tokenizers \
    openai rich ipywidgets pandas matplotlib psutil

In [ ]:
import subprocess, json, time, os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import openvino as ov
import openvino_genai as ov_genai
import psutil
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ── Lunar Lake system check ────────────────────────────────────────────
def check_lunar_lake():
    checks = {}

    # CPU model
    try:
        cpu = subprocess.check_output("cat /proc/cpuinfo | grep 'model name' | head -1",
                                       shell=True).decode().split(":")[-1].strip()
        checks["CPU"] = cpu
    except:
        checks["CPU"] = "Unknown"

    # Kernel
    try:
        checks["Kernel"] = subprocess.check_output("uname -r", shell=True).decode().strip()
    except:
        checks["Kernel"] = "Unknown"

    # Intel GPU driver
    try:
        lz = subprocess.check_output("dpkg -l intel-level-zero-gpu 2>/dev/null | tail -1",
                                      shell=True).decode().strip()
        checks["Level Zero GPU"] = lz.split()[-1] if lz else "NOT INSTALLED"
    except:
        checks["Level Zero GPU"] = "NOT INSTALLED"

    # NPU driver version (critical for Lunar Lake)
    try:
        npu_pkg = subprocess.check_output("dpkg -l intel-npu-driver 2>/dev/null | tail -1",
                                           shell=True).decode().strip()
        checks["NPU Driver"] = npu_pkg.split()[-1] if npu_pkg else "NOT INSTALLED"
    except:
        checks["NPU Driver"] = "NOT INSTALLED"

    # NPU device node
    npu_devs = list(Path("/dev/accel").glob("accel*")) if Path("/dev/accel").exists() else []
    checks["NPU Device"] = str(npu_devs[0]) if npu_devs else "✗ Not found"

    # Total and available RAM (critical for unified memory)
    mem = psutil.virtual_memory()
    checks["Total RAM"] = f"{mem.total/1e9:.1f} GB"
    checks["Available RAM"] = f"{mem.available/1e9:.1f} GB"

    # OpenVINO
    checks["OpenVINO"] = ov.__version__
    checks["OpenVINO GenAI"] = ov_genai.__version__

    return checks

info = check_lunar_lake()
table = Table(title="🔍 Lunar Lake System Info", show_header=True)
table.add_column("Component", style="cyan", min_width=22)
table.add_column("Value", style="white")
for k, v in info.items():
    ok = "✗" not in str(v) and "NOT" not in str(v)
    style = "green" if ok else "red"
    table.add_row(k, f"[{style}]{v}[/{style}]")
console.print(table)

In [ ]:
# Device discovery and Lunar Lake-specific warnings
core = ov.Core()
available_devices = core.available_devices

table = Table(title="OpenVINO Device Discovery")
table.add_column("Device", style="cyan")
table.add_column("Full Name", style="green")
table.add_column("Status")
for d in available_devices:
    try:
        name = core.get_property(d, "FULL_DEVICE_NAME")
        status = "[green]✓ Ready[/green]"
    except:
        name = "N/A"
        status = "[yellow]Limited[/yellow]"
    table.add_row(d, name, status)
console.print(table)

# Unified memory warning
ram_gb = psutil.virtual_memory().total / 1e9
if ram_gb < 16:
    console.print(Panel(
        f"[yellow]Only {ram_gb:.0f}GB RAM detected.\n"
        "Lunar Lake uses unified memory shared between CPU and GPU.\n"
        "16GB+ recommended for Qwen3-8B. 32GB+ for 30B models.\n"
        "Models >16GB prompts may require setting MAX_PROMPT_LEN carefully.[/yellow]",
        title="⚠ Unified Memory Warning", border_style="yellow"
    ))

# NPU driver version check
npu_driver = info.get("NPU Driver", "")
if "32.0.100" in npu_driver:
    driver_num = npu_driver.split(".")
    if len(driver_num) >= 4 and int(driver_num[-1]) < 3104:
        console.print(Panel(
            "[yellow]NPU driver version may be too old for stable LLM inference.\n"
            "Minimum recommended: 32.0.100.3104\n"
            "Download: github.com/intel/linux-npu-driver/releases[/yellow]",
            title="⚠ NPU Driver Warning", border_style="yellow"
        ))

## 2. Configuration & Model Preparation

In [ ]:
# Load config from notebook 1
CONFIG_FILE = Path("agent_config.json")
if CONFIG_FILE.exists():
    with open(CONFIG_FILE) as f:
        cfg = json.load(f)
    console.print("[green]✓ Loaded config from notebook 1[/green]")
else:
    cfg = {
        "model_id": "Qwen/Qwen3-8B",
        "model_name": "Qwen3-8B",
        "model_path": "Qwen3-8B-int4-ov",
        "tool_parser": "hermes3",
        "reasoning_parser": "qwen3",
        "ovms_port": 8000,
    }
    console.print("[yellow]No config found — using defaults[/yellow]")

MODEL_NAME = cfg["model_name"]
MODEL_PATH = Path(cfg["model_path"])
OVMS_PORT  = cfg["ovms_port"]

# NPU-specific model path (channel-wise quantization)
NPU_MODEL_PATH = Path(str(MODEL_PATH).replace("-int4-ov", "-int4-cw-ov"))

N_WARMUP   = 2
N_REPEATS  = 5
MAX_NEW_TOKENS = 256

DEVICES_TO_BENCH = [d for d in ["CPU", "GPU", "NPU"] if d in available_devices]

console.print(f"Model:     {MODEL_NAME}")
console.print(f"GPU model: {MODEL_PATH}")
console.print(f"NPU model: {NPU_MODEL_PATH} ({'found' if NPU_MODEL_PATH.exists() else 'not found — will export'})")
console.print(f"Devices:   {DEVICES_TO_BENCH}")

In [ ]:
# Export NPU-optimized model if not present
# Lunar Lake NPU requires: INT4, sym, ratio=1.0, group-size=-1

if "NPU" in DEVICES_TO_BENCH and not NPU_MODEL_PATH.exists():
    console.print(f"[yellow]Exporting NPU-optimized model to {NPU_MODEL_PATH}...[/yellow]")
    console.print("[dim]Lunar Lake NPU requires: --sym --ratio 1.0 --group-size -1[/dim]")

    npu_export_cmd = (
        f"optimum-cli export openvino "
        f"--model {cfg['model_id']} "
        f"--task text-generation-with-past "
        f"--weight-format int4 "
        f"--sym "
        f"--ratio 1.0 "
        f"--group-size -1 "
        f"{NPU_MODEL_PATH}"
    )
    console.print(f"[dim]Running: {npu_export_cmd}[/dim]")
    ret = subprocess.run(npu_export_cmd, shell=True)
    if ret.returncode == 0:
        console.print("[green]✓ NPU model exported[/green]")
    else:
        console.print("[red]✗ Export failed[/red]")
elif NPU_MODEL_PATH.exists():
    console.print(f"[green]✓ NPU model already at {NPU_MODEL_PATH}[/green]")

# Alternatively, use pre-optimized models from OpenVINO HuggingFace collection
console.print("\n[dim]Tip: Pre-optimized NPU models available at:[/dim]")
console.print("[dim]  https://huggingface.co/collections/OpenVINO/llms-optimized-for-npu[/dim]")
console.print("[dim]  e.g. huggingface-cli download OpenVINO/Qwen3-8B-int4-cw-ov[/dim]")

## 3. Lunar Lake-Specific NPU Configuration

Lunar Lake's NPU4 has unique properties that must be set correctly.

In [ ]:
# Lunar Lake NPU4 properties and constraints reference
LUNAR_LAKE_NPU_CONFIG = {
    # Maximum context window on NPU
    # Context = MAX_PROMPT_LEN + MIN_RESPONSE_LEN
    "MAX_PROMPT_LEN": "1024",     # max input tokens
    "MIN_RESPONSE_LEN": "512",    # reserved output tokens

    # Cache for compiled NPU model (avoids recompilation each run)
    # First run may take several minutes; subsequent runs use cache
    "CACHE_DIR": ".npu_cache",

    # Prefill chunk size (OpenVINO 2025.3+)
    # Smaller chunks reduce TTFT for short prompts
    "NPUW_LLM_PREFILL_CHUNK_SIZE": "512",
}

table = Table(title="Lunar Lake NPU4 Configuration")
table.add_column("Property", style="cyan")
table.add_column("Value", style="green")
table.add_column("Notes", style="dim")

notes = {
    "MAX_PROMPT_LEN": "Max input tokens before truncation",
    "MIN_RESPONSE_LEN": "Tokens reserved for output",
    "CACHE_DIR": "Compiled model cache (speeds up reload)",
    "NPUW_LLM_PREFILL_CHUNK_SIZE": "Chunk size for dynamic prompt support",
}
for k, v in LUNAR_LAKE_NPU_CONFIG.items():
    table.add_row(k, str(v), notes.get(k, ""))
console.print(table)

# NPU-specific constraints
console.print(Panel(
    "[bold]Lunar Lake NPU4 Constraints:[/bold]\n"
    "• INT4 quantization only (not INT8 or FP16)\n"
    "• Beam search NOT supported (use greedy or multinomial)\n"
    "• No concurrent batching — requests processed sequentially\n"
    "• NF4 format supported on LNL and newer (Lunar Lake+)\n"
    "• First run compiles model — takes 2–5 min; cached after\n"
    "• RAM: 8B model needs ~10GB; ensure enough free unified memory",
    title="NPU Constraints", border_style="yellow"
))

## 4. Throughput & Latency Benchmark

In [ ]:
BENCH_PROMPTS = [
    "Write a Python function to implement merge sort with step-by-step comments.",
    "Implement a thread-safe queue in Python using locks.",
    "Write a Python decorator for retrying failed functions with exponential backoff.",
    "Implement a simple tokenizer for arithmetic expressions in Python.",
    "Write a Python function to find all prime numbers up to N using the Sieve of Eratosthenes.",
]

def measure_generation(pipe, prompt, max_new_tokens=256):
    """Measure TTFT and decode throughput."""
    token_times = []
    start = time.perf_counter()

    def cb(token):
        token_times.append(time.perf_counter())
        return False

    pipe.generate(prompt, max_new_tokens=max_new_tokens, streamer=cb)
    end = time.perf_counter()

    n = len(token_times)
    ttft = (token_times[0] - start) * 1000 if token_times else None
    if n > 1:
        decode_time = token_times[-1] - token_times[0]
        tps = (n - 1) / decode_time if decode_time > 0 else 0
    else:
        tps = 0

    return {
        "ttft_ms": round(ttft, 1) if ttft else None,
        "throughput_tps": round(tps, 2),
        "total_tokens": n,
        "total_time_s": round(end - start, 3),
    }

console.print("[green]✓ Benchmark functions ready[/green]")

In [ ]:
all_results = {}
memory_usage = {}

for device in DEVICES_TO_BENCH:
    console.print(f"\n[bold cyan]═══ Benchmarking on {device} (Lunar Lake) ═══[/bold cyan]")

    # Use NPU-specific model for NPU device
    if device == "NPU":
        model_path = NPU_MODEL_PATH if NPU_MODEL_PATH.exists() else MODEL_PATH
        props = LUNAR_LAKE_NPU_CONFIG.copy()
        console.print(f"  Model: {model_path}")
        console.print(f"  [dim]First NPU run may take 2–5 min for compilation[/dim]")
    else:
        model_path = MODEL_PATH
        props = {}

    if not model_path.exists():
        console.print(f"  [red]Model not found at {model_path} — skipping[/red]")
        continue

    # Measure RAM before loading
    ram_before = psutil.virtual_memory().used / 1e9

    try:
        load_start = time.perf_counter()
        pipe = ov_genai.LLMPipeline(str(model_path), device, **props)
        load_time = time.perf_counter() - load_start

        ram_after = psutil.virtual_memory().used / 1e9
        ram_used = ram_after - ram_before
        memory_usage[device] = round(ram_used, 2)

        console.print(f"  Load time: {load_time:.1f}s | RAM delta: {ram_used:.1f} GB")

        # Warmup
        console.print(f"  Warming up ({N_WARMUP} runs)...")
        for _ in range(N_WARMUP):
            pipe.generate(BENCH_PROMPTS[0], max_new_tokens=32)

        # Benchmark runs
        device_results = []
        for i, prompt in enumerate(BENCH_PROMPTS[:N_REPEATS]):
            console.print(f"  Run {i+1}/{N_REPEATS}: ", end="")
            r = measure_generation(pipe, prompt, MAX_NEW_TOKENS)
            device_results.append(r)
            console.print(f"TTFT={r['ttft_ms']}ms  {r['throughput_tps']:.1f} tok/s  ({r['total_tokens']} tokens)")

        all_results[device] = device_results
        del pipe

    except Exception as e:
        console.print(f"  [red]✗ Error on {device}: {e}[/red]")
        if device == "NPU" and "memory" in str(e).lower():
            console.print("  [yellow]NPU OOM — try setting DISABLE_OPENVINO_GENAI_NPU_L0=1[/yellow]")
        all_results[device] = []

## 5. Agent Task Benchmark

In [ ]:
# Agent task benchmark (OVMS must be running)
from openai import OpenAI
import tempfile

AGENT_TASKS = [
    "Write a Python class for a min-heap and test insert and extract_min.",
    "Write a Python function for longest common subsequence and test it.",
    "Implement a rate limiter in Python using a token bucket algorithm. Test it.",
]

TOOLS_MINIMAL = [
    {
        "type": "function",
        "function": {
            "name": "run_python",
            "description": "Run Python code and return output.",
            "parameters": {
                "type": "object",
                "properties": {"code": {"type": "string"}},
                "required": ["code"]
            }
        }
    }
]

def run_python_tool(code, timeout=15):
    with tempfile.NamedTemporaryFile(suffix=".py", mode="w", delete=False) as f:
        f.write(code); fname = f.name
    try:
        r = subprocess.run([sys.executable, fname], capture_output=True, text=True, timeout=timeout)
        return (r.stdout + ("\n" + r.stderr if r.stderr else "")).strip()
    except subprocess.TimeoutExpired:
        return "[TIMEOUT]"
    finally:
        os.unlink(fname)

def run_agent_task(task, client, model_name, max_turns=8):
    messages = [
        {"role": "system", "content": "You are a coding expert. Write and test Python code."},
        {"role": "user", "content": task}
    ]
    start = time.perf_counter()
    turns = tool_uses = prompt_tokens = completion_tokens = 0

    while turns < max_turns:
        turns += 1
        resp = client.chat.completions.create(
            model=model_name, messages=messages, tools=TOOLS_MINIMAL,
            max_tokens=1024,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}}
        )
        msg = resp.choices[0].message
        if resp.usage:
            prompt_tokens += resp.usage.prompt_tokens
            completion_tokens += resp.usage.completion_tokens

        msg_dict = {"role": "assistant", "content": msg.content or ""}
        if msg.tool_calls:
            msg_dict["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]
        messages.append(msg_dict)

        if not msg.tool_calls:
            break
        for tc in msg.tool_calls:
            tool_uses += 1
            args = json.loads(tc.function.arguments)
            result = run_python_tool(args.get("code", ""))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    elapsed = time.perf_counter() - start
    return {
        "task": task[:60] + "...",
        "turns": turns,
        "tool_uses": tool_uses,
        "total_time_s": round(elapsed, 2),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "tps": round(completion_tokens / elapsed, 2) if elapsed > 0 else 0,
    }

# Run if OVMS is up
try:
    client = OpenAI(base_url=f"http://localhost:{OVMS_PORT}/v3", api_key="unused")
    client.models.list()

    agent_results = []
    for task in AGENT_TASKS:
        console.print(f"\n[cyan]Task:[/cyan] {task[:70]}...")
        r = run_agent_task(task, client, MODEL_NAME)
        agent_results.append(r)
        console.print(f"  {r['total_time_s']}s | {r['turns']} turns | "
                      f"{r['tool_uses']} tool calls | {r['tps']} tok/s")

    agent_df = pd.DataFrame(agent_results)
    console.print("\n[bold]Agent Benchmark Summary:[/bold]")
    console.print(agent_df[["task", "total_time_s", "turns", "tool_uses", "tps"]].to_string(index=False))

except Exception as e:
    console.print(f"[yellow]OVMS not reachable ({e}) — start it from notebook 01[/yellow]")
    agent_results = []

## 6. Results Visualization

In [ ]:
# Summary table
summary_rows = []
for device, runs in all_results.items():
    if not runs: continue
    ttfts = [r["ttft_ms"] for r in runs if r["ttft_ms"]]
    tputs = [r["throughput_tps"] for r in runs if r["throughput_tps"] > 0]
    summary_rows.append({
        "Device": device,
        "Avg TTFT (ms)": round(sum(ttfts)/len(ttfts), 1) if ttfts else None,
        "Avg Throughput (tok/s)": round(sum(tputs)/len(tputs), 2) if tputs else None,
        "Peak Throughput (tok/s)": round(max(tputs), 2) if tputs else None,
        "RAM Delta (GB)": memory_usage.get(device, "N/A"),
    })

if summary_rows:
    df = pd.DataFrame(summary_rows)

    table = Table(title=f"📊 Lunar Lake Benchmark — {MODEL_NAME}", show_lines=True)
    for col in df.columns:
        table.add_column(col, style="cyan" if col == "Device" else "white")
    for _, row in df.iterrows():
        table.add_row(*[str(v) for v in row])
    console.print(table)

In [ ]:
if summary_rows:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Lunar Lake (Xe2 Arc 140V / NPU4) — {MODEL_NAME} INT4",
                 fontsize=13, fontweight="bold")

    colors = {"CPU": "#4A90D9", "GPU": "#27AE60", "NPU": "#8E44AD"}
    devices = df["Device"].tolist()
    bar_colors = [colors.get(d, "#888") for d in devices]

    # 1. Throughput
    tput_vals = df["Avg Throughput (tok/s)"].tolist()
    bars = axes[0].bar(devices, tput_vals, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.2)
    axes[0].set_title("Decode Throughput"); axes[0].set_ylabel("Tokens / second")
    for bar, val in zip(bars, tput_vals):
        if val: axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                              f"{val:.1f}", ha="center", fontweight="bold")
    axes[0].spines[["top", "right"]].set_visible(False)

    # 2. TTFT
    ttft_vals = df["Avg TTFT (ms)"].tolist()
    bars2 = axes[1].bar(devices, ttft_vals, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.2)
    axes[1].set_title("Time to First Token"); axes[1].set_ylabel("ms (lower is better)")
    for bar, val in zip(bars2, ttft_vals):
        if val: axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                              f"{val:.0f}ms", ha="center", fontweight="bold")
    axes[1].spines[["top", "right"]].set_visible(False)

    # 3. Memory usage
    mem_vals = [memory_usage.get(d, 0) for d in devices]
    bars3 = axes[2].bar(devices, mem_vals, color=bar_colors, width=0.5, edgecolor="white", linewidth=1.2)
    axes[2].set_title("RAM Usage (Unified Memory)")
    axes[2].set_ylabel("GB (shared CPU+GPU pool)")
    for bar, val in zip(bars3, mem_vals):
        if val: axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                              f"{val:.1f}GB", ha="center", fontweight="bold")
    axes[2].spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    out_path = "lunarlake_benchmark.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    console.print(f"[green]Chart saved to {out_path}[/green]")

## 7. Panther Lake vs Lunar Lake Comparison

In [ ]:
# Load Panther Lake results if available for side-by-side comparison
pl_file = Path("pantherlake_results.json")

if pl_file.exists():
    with open(pl_file) as f:
        pl_data = json.load(f)

    # Build comparison dataframe
    comp_rows = []

    for platform, results_dict in [("Lunar Lake", all_results),
                                    ("Panther Lake", pl_data.get("genai_pipeline_results", {}))]:
        for device, runs in results_dict.items():
            if not runs: continue
            tputs = [r["throughput_tps"] for r in runs if r["throughput_tps"] > 0]
            ttfts = [r["ttft_ms"] for r in runs if r["ttft_ms"]]
            if tputs:
                comp_rows.append({
                    "Platform": platform,
                    "Device": device,
                    "Avg Throughput": round(sum(tputs)/len(tputs), 2),
                    "Avg TTFT (ms)": round(sum(ttfts)/len(ttfts), 1) if ttfts else None,
                })

    if comp_rows:
        comp_df = pd.DataFrame(comp_rows)

        table = Table(title=f"🏁 Panther Lake vs Lunar Lake — {MODEL_NAME}", show_lines=True)
        table.add_column("Platform", style="cyan")
        table.add_column("Device", style="green")
        table.add_column("Avg Throughput (tok/s)")
        table.add_column("Avg TTFT (ms)")
        for _, row in comp_df.iterrows():
            table.add_row(*[str(v) for v in row])
        console.print(table)

        # Side-by-side bar chart
        fig, ax = plt.subplots(figsize=(12, 5))
        fig.suptitle(f"Panther Lake vs Lunar Lake — Throughput (tok/s) — {MODEL_NAME}",
                     fontweight="bold")

        import numpy as np
        platforms = comp_df["Platform"].unique()
        device_types = comp_df["Device"].unique()
        x = np.arange(len(device_types))
        width = 0.35

        pl_colors = {"Panther Lake": "#E74C3C", "Lunar Lake": "#3498DB"}
        for i, platform in enumerate(platforms):
            pdata = comp_df[comp_df["Platform"] == platform]
            vals = []
            for d in device_types:
                row = pdata[pdata["Device"] == d]
                vals.append(float(row["Avg Throughput"].values[0]) if len(row) else 0)
            offset = width * (i - len(platforms)/2 + 0.5)
            bars = ax.bar(x + offset, vals, width, label=platform,
                          color=pl_colors.get(platform, "#888"), alpha=0.85)
            for bar, val in zip(bars, vals):
                if val > 0:
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                            f"{val:.1f}", ha="center", fontsize=9, fontweight="bold")

        ax.set_xticks(x); ax.set_xticklabels(device_types)
        ax.set_ylabel("Tokens / second")
        ax.legend()
        ax.spines[["top", "right"]].set_visible(False)
        plt.tight_layout()
        plt.savefig("comparison_pantherlake_vs_lunarlake.png", dpi=150, bbox_inches="tight")
        plt.show()
        console.print("[green]Comparison chart saved[/green]")
else:
    console.print("[yellow]Run notebook 02 first to generate Panther Lake results for comparison[/yellow]")

## 8. Save Results

In [ ]:
results_out = {
    "platform": "Lunar Lake",
    "model": MODEL_NAME,
    "kernel": info.get("Kernel"),
    "openvino": info.get("OpenVINO"),
    "ram_gb": info.get("Total RAM"),
    "memory_usage_per_device_gb": memory_usage,
    "genai_pipeline_results": all_results,
    "agent_task_results": agent_results if 'agent_results' in dir() else [],
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}

out_file = Path("lunarlake_results.json")
out_file.write_text(json.dumps(results_out, indent=2))
console.print(f"[green]✓ Results saved to {out_file}[/green]")

console.print(Panel(
    "[bold]Lunar Lake Benchmark Complete[/bold]\n\n"
    "Key Lunar Lake takeaways:\n"
    "• NPU4: best power efficiency, great for ≤8B models, frees GPU\n"
    "• GPU (Xe2 Arc 140V): best throughput; uses unified memory\n"
    "• Unified memory: watch total consumption — GPU and CPU share it\n"
    "• NPU driver maturity: LNL is the most stable NPU platform today\n\n"
    "Files generated:\n"
    "  lunarlake_results.json\n"
    "  lunarlake_benchmark.png\n"
    "  comparison_pantherlake_vs_lunarlake.png (if PL results exist)",
    title="Summary", border_style="green"
))